# APIs de LLM — práctica (Sesión 1)

**OpenAI:** **2a** — Responses API (referencia principal). **2b** — Chat Completions (multi‑proveedor / LiteLLM). **3** — Anthropic Messages API. **4** — **Google Gemini** con el SDK **`google-genai`** (Gemini Developer API con API key; el paquete antiguo `google-generativeai` está deprecado). **5** — **Tokens y tokenización** (`tiktoken` + comparativa práctica; `count_tokens` en Gemini).

Python 3.11+, `uv sync`, claves en Colab (secretos) o entorno. **No subas API keys a Git.**


## OpenAI: dos APIs, una recomendación (criterio del máster)

- **Responses API** (`client.responses.create`): la interfaz **más reciente y recomendada** para proyectos nuevos. Unifica capacidades, usa `instructions` + `input`, `output_text`, gestión de estado (`previous_response_id`) y herramientas nativas. **En este programa es la API principal;** los ejercicios 2a siguen esta estructura.
- **Chat Completions** (`client.chat.completions.create`): API **anterior**, soportada indefinidamente pero **ya no recomendada** para desarrollos nuevos solo en OpenAI. Su patrón `messages` con roles (`system`, `user`, `assistant`) es el que comparten **Anthropic, Google, Mistral**, etc., y el que aparecerá al trabajar **abstracción de proveedores** y agregadores como **LiteLLM**.

**Colab:** `%pip install -q openai anthropic google-genai tiktoken` (cada ejercicio puede repetir `%pip` mínimo si prefieres). Secretos `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, `GEMINI_API_KEY` con acceso al notebook. **Local:** `uv sync` (incluye `tiktoken`) y `export ...` según proveedor.

## Ejercicio 2a — Responses API: llamada completa y piezas

### 1. Llamada completa (diferencias frente a Chat Completions)

- **`instructions`**: rol, reglas y restricciones (equivalente al *system prompt*), **parámetro de primer nivel**, no es un mensaje más en el hilo.
- **`input`**: entrada del usuario — **string** en un solo turno o **lista de mensajes** con roles (`user`, `assistant`, `developer`) para multi-turno o más control.
- **Salida:** `response.output_text` (atajo del SDK); en casos avanzados (herramientas, varios bloques) se recorre `response.output`.

### 2. Contexto multi-turno

- **Manual:** incluir todo el historial en `input` como lista (igual que reconstruir mensajes en Chat Completions).
- **`previous_response_id`:** encadena la respuesta anterior; suele usarse con **`store=True`** para que OpenAI guarde y recupere contexto (implicaciones de **privacidad** si hay datos sensibles — entonces `store=False` e historial manual).

### 3. Parámetros clave

`model`, `temperature`, **`max_output_tokens`** (tope de generación; si se alcanza, `status` puede ser `incomplete`), **`store`** (persistencia en servidores de OpenAI).

### 4. Respuesta: contenido, estado y metadatos

- **`status`:** `completed` (esperado), `incomplete` (revisar `incomplete_details`, p. ej. límite de salida), `failed` (`error`).
- **`usage`:** `input_tokens`, `output_tokens`, `total_tokens`; en modelos con razonamiento, detalle en `output_tokens_details`.

### 5. Coste orientativo

Los precios cambian: [API pricing](https://openai.com/api-pricing). El ejemplo en código usa tarifas de ejemplo para `gpt-4o-mini` (revisa la web antes de usar en producción).

### 6. Errores

El SDK expone excepciones tipadas (`AuthenticationError`, `RateLimitError`, `BadRequestError`, …). El código siguiente incluye un **try/except** mínimo y comprobación de `status`.

Documentación: [Responses](https://platform.openai.com/docs/guides/text).

In [ ]:
# %pip install -q openai

import os
from getpass import getpass

from openai import (
    APIConnectionError,
    AuthenticationError,
    BadRequestError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)


def obtener_api_key_openai() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        return key
    return getpass("OPENAI_API_KEY: ").strip()


# Precios orientativos USD por 1M tokens (gpt-4o-mini) — actualiza desde la web de OpenAI
PRICING_GPT4O_MINI = {"input": 0.15, "output": 0.60}

api_key = obtener_api_key_openai()
if not api_key:
    raise ValueError("Falta OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

instructions = """Eres un consultor senior de estimación de proyectos software con amplia experiencia.

Reglas:
- Responde siempre en español.
- Usa terminología técnica sin simplificar en exceso.
- Si ofreces una estimación temporal, indica un rango (optimista / pesimista).
- Si faltan datos para estimar con rigor, pregunta antes de suponer.
- Redacta en prosa; evita listas con viñetas salvo que el usuario las pida."""

input_usuario = (
    "¿Qué factores debo considerar al estimar un proyecto de migración de base de datos?"
)

try:
    respuesta = client.responses.create(
        model=MODEL,
        instructions=instructions,
        input=input_usuario,
        temperature=0.3,
        max_output_tokens=500,
        store=False,
    )
except AuthenticationError:
    print("401: revisa OPENAI_API_KEY (válida y activa).")
    raise
except RateLimitError:
    print("429: límite de tasa o crédito; espera o revisa facturación.")
    raise
except BadRequestError as e:
    print("400: petición mal formada:", getattr(e, "message", e))
    raise
except (APIConnectionError, InternalServerError) as e:
    print("Red o error en servidor OpenAI:", e)
    raise

print("--- Contenido (output_text) ---")
print(respuesta.output_text)

print("--- Estado ---")
print("status:", respuesta.status)
if respuesta.status == "incomplete" and respuesta.incomplete_details:
    print("incomplete_details:", respuesta.incomplete_details)
if respuesta.status == "failed" and respuesta.error:
    print("error:", respuesta.error)

print("--- Metadatos ---")
print("id:", respuesta.id)
print("model (snapshot real):", respuesta.model)

if respuesta.usage:
    u = respuesta.usage
    print("usage:", u.model_dump())
    inp = u.input_tokens
    out = u.output_tokens
    p = PRICING_GPT4O_MINI
    coste = (inp / 1_000_000) * p["input"] + (out / 1_000_000) * p["output"]
    print(f"coste aprox. (USD): ${coste:.6f}")

# --- Opcional: mismo modelo con input como lista (multi-turno manual) ---
# respuesta_2 = client.responses.create(
#     model=MODEL,
#     instructions=instructions,
#     input=[
#         {"role": "user", "content": "¿Qué es una API REST?"},
#         {"role": "assistant", "content": "Una API REST es una interfaz basada en HTTP y recursos..."},
#         {"role": "user", "content": "¿En qué se diferencia de GraphQL?"},
#     ],
#     temperature=0.3,
#     max_output_tokens=400,
#     store=False,
# )
# print(respuesta_2.output_text)

# --- Opcional: encadenar con previous_response_id (requiere store=True en la 1ª llamada) ---
# Ver guía del máster; implica almacenamiento en OpenAI.

## Ejercicio 2b — Chat Completions: patrón `messages` (legacy OpenAI, estándar multi‑proveedor)

La **Chat Completions API** (`client.chat.completions.create`) es la API **anterior** de OpenAI: **sigue soportada**, pero **ya no es la recomendada** para desarrollos nuevos *solo* en OpenAI.

Su estructura de **`messages`** con roles **`system`**, **`user`**, **`assistant`** es el patrón que comparten **Anthropic, Google, Mistral** y muchos agregadores; por eso **sigue siendo imprescindible entenderla**. La verás al trabajar **abstracción de proveedores** y herramientas como **LiteLLM** (interfaz tipo Chat Completions).

### Desglose del `system` (equivalente conceptual a `instructions` en Responses)

1. **Rol:** quién es el modelo (“Eres un consultor senior…”).
2. **Instrucciones operativas:** qué hacer y cómo (idioma, tono, formato).
3. **Restricciones:** qué evitar (inventar datos, viñetas innecesarias, etc.).

### Diferencias clave respecto a Responses (resumen)

| Concepto | Responses API | Chat Completions |
|----------|---------------|------------------|
| Rol / reglas globales | `instructions` (top-level) | mensaje `role: "system"` |
| Entrada usuario (1 turno) | `input` (string o lista) | `messages` con `user` |
| Texto principal de salida | `output_text` | `choices[0].message.content` |
| Tope de salida | `max_output_tokens` | `max_tokens` |
| Estado explícito | `status`, `incomplete_details` | `finish_reason` en la choice |

Documentación: [Chat Completions](https://platform.openai.com/docs/guides/text-generation).

In [ ]:
# %pip install -q openai

import os
from getpass import getpass

from openai import (
    APIConnectionError,
    AuthenticationError,
    BadRequestError,
    InternalServerError,
    OpenAI,
    RateLimitError,
)


def obtener_api_key_openai() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("OPENAI_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("OPENAI_API_KEY", "").strip()
    if key:
        return key
    return getpass("OPENAI_API_KEY: ").strip()


PRICING_GPT4O_MINI = {"input": 0.15, "output": 0.60}

api_key = obtener_api_key_openai()
if not api_key:
    raise ValueError("Falta OPENAI_API_KEY")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4o-mini"

system_content = """Eres un consultor senior de estimación de proyectos software con amplia experiencia.

Reglas:
- Responde siempre en español.
- Usa terminología técnica sin simplificar en exceso.
- Si ofreces una estimación temporal, indica un rango (optimista / pesimista).
- Si faltan datos para estimar con rigor, pregunta antes de suponer.
- Redacta en prosa; evita listas con viñetas salvo que el usuario las pida."""

user_content = (
    "¿Qué factores debo considerar al estimar un proyecto de migración de base de datos?"
)

try:
    respuesta = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_content},
            {"role": "user", "content": user_content},
        ],
        temperature=0.3,
        max_tokens=500,
    )
except AuthenticationError:
    print("401: revisa OPENAI_API_KEY.")
    raise
except RateLimitError:
    print("429: límite de tasa o crédito.")
    raise
except BadRequestError as e:
    print("400:", getattr(e, "message", e))
    raise
except (APIConnectionError, InternalServerError) as e:
    print("Red o servidor:", e)
    raise

choice = respuesta.choices[0]
msg = choice.message.content

print("--- Contenido (choices[0].message.content) ---")
print(msg)

print("--- Metadatos ---")
print("finish_reason:", choice.finish_reason)
print("id:", respuesta.id)
print("model:", respuesta.model)

if respuesta.usage:
    u = respuesta.usage
    print("usage:", u.model_dump())
    inp = u.prompt_tokens
    out = u.completion_tokens
    p = PRICING_GPT4O_MINI
    coste = (inp / 1_000_000) * p["input"] + (out / 1_000_000) * p["output"]
    print(f"coste aprox. (USD): ${coste:.6f}")

# Inspección opcional del JSON completo (útil en depuración):
# print(respuesta.model_dump_json(indent=2)[:2000])

## Ejercicio 3 — Anthropic Messages API (`messages.create`)

### Contexto: una sola API

Anthropic expone **una única** interfaz para texto generado: la **Messages API**. A diferencia de OpenAI (Responses + Chat Completions en paralelo), aquí **toda** la funcionalidad consolidada pasa por `client.messages.create`.

El patrón recuerda a **Chat Completions**: array `messages` con roles y respuesta estructurada. Las diferencias clave frente a OpenAI: **`system` en parámetro aparte** (como `instructions` en Responses), **`max_tokens` obligatorio**, salida como **bloques** en `content` (no hay `output_text`), y detalles de parámetros (`temperature` 0–1, `top_k`, etc.).

**Colab:** `%pip install -q anthropic`, secreto `ANTHROPIC_API_KEY` con acceso al notebook. **Local:** `export ANTHROPIC_API_KEY=...`.

Documentación: [Messages](https://docs.anthropic.com/en/api/messages).

### 1. Llamada completa

- `model`, `system`, `messages`, **`max_tokens` (obligatorio)**, `temperature`.
- Texto típico: `response.content[0].text` o iterar bloques (herramientas, *extended thinking*, etc.).

### 2. `system`: rol, instrucciones y restricciones

Igual que `instructions` en OpenAI Responses: **primer nivel**, separado del hilo. Puede ser **string** o **lista de bloques** (`type: text`, `cache_control` para *prompt caching* en sesiones posteriores). En los ejercicios basta el string.

### 3. `messages`: conversación y roles

Solo **`user`** y **`assistant`** dentro del array: **no** existe `system` dentro de `messages`.

**Alternancia estricta:** el primer mensaje debe ser `user`; no pueden ir dos `user` seguidos. Para varias piezas de contexto en un turno, usa un solo `user` con **bloques** de `content` o texto concatenado.

La API es **stateless**: no hay equivalente a `previous_response_id`; el historial se **reenvía entero** en cada llamada (impacto en **tokens** y coste). Anthropic mitiga repetición con **prompt caching** (sesiones posteriores).

### 4. Parámetros esenciales y otros

| Parámetro | Notas |
|-----------|--------|
| `max_tokens` | **Obligatorio**. Si se agota, `stop_reason == "max_tokens"`. |
| `temperature` | 0.0–**1.0** (no hasta 2.0 como OpenAI). |
| `top_p` / `top_k` | No mezclar `temperature` y `top_p`; `top_k` es propio de Anthropic. |
| `stop_sequences` | Cortan la generación si el modelo las emite. |

*Extended thinking* (modelos compatibles): parámetro `thinking` con presupuesto de tokens; la respuesta puede incluir bloques `thinking` facturados como salida.

### 5. Respuesta: contenido, `stop_reason`, metadatos

- **`content`**: lista de bloques; lo habitual es un bloque `text`.
- **`stop_reason`:** `end_turn` (ok), `max_tokens` (truncado), `stop_sequence`, `tool_use`, etc.
- **`id`**, **`model`** (coincide con el id de modelo pedido), **`type`**, **`role`**. No viene **timestamp** en la respuesta: regístralo en tu código si lo necesitas.

### 6. `usage` y coste

`input_tokens` y `output_tokens`; **no** hay `total_tokens` en la respuesta — súmalos tú. Con *prompt caching* aparecen campos extra (`cache_creation_input_tokens`, `cache_read_input_tokens`, …).

Precios: [Anthropic pricing](https://www.anthropic.com/pricing). El código usa tarifas de **ejemplo** para `claude-haiku-4-5-20251001` (revísalas antes de producción).

### 7. Errores y reintentos del SDK

Excepciones tipadas (`AuthenticationError`, `RateLimitError`, `BadRequestError`, …). El cliente **reintenta** por defecto errores transitorios (p. ej. 429, 5xx); configurable con `Anthropic(max_retries=...)`.

### 8. Equivalencia rápida con OpenAI

| Concepto | OpenAI Responses | OpenAI Chat | Anthropic Messages |
|----------|------------------|-------------|---------------------|
| Rol / reglas globales | `instructions` | mensaje `system` | parámetro `system` |
| Hilo conversación | `input` (str o lista) | `messages` | `messages` (solo user/assistant) |
| Texto principal | `output_text` | `choices[0].message.content` | bloques `content` (p. ej. `.text`) |
| Tope salida | `max_output_tokens` | `max_tokens` (opcional) | **`max_tokens` (obligatorio)** |
| Fin / estado | `status`, … | `finish_reason` | `stop_reason` |
| Historial multi-turno | manual o `previous_response_id`+`store` | manual | **solo manual** |

### 9. Claves y `401`

- Clave en [Anthropic Console](https://console.anthropic.com/) (`sk-ant-...`).
- Secreto Colab: nombre exacto `ANTHROPIC_API_KEY`, sin espacios al pegar.


In [ ]:
# %pip install -q anthropic

import os
from getpass import getpass

from anthropic import (
    Anthropic,
    APIConnectionError,
    AuthenticationError,
    BadRequestError,
    InternalServerError,
    RateLimitError,
)


def obtener_api_key_anthropic() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("ANTHROPIC_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    key = os.environ.get("ANTHROPIC_API_KEY", "").strip()
    if key:
        return key
    return getpass("ANTHROPIC_API_KEY: ").strip()


# USD por 1M tokens (ejemplo Haiku 4.5 — actualiza en docs.anthropic.com / pricing)
PRICING_HAIKU = {"input": 1.00, "output": 5.00}


def texto_desde_content(content) -> str:
    partes = []
    for bloque in content:
        if getattr(bloque, "type", None) == "text" and hasattr(bloque, "text"):
            partes.append(bloque.text)
    return "".join(partes)


api_key = obtener_api_key_anthropic().strip()
if not api_key:
    raise ValueError("Falta ANTHROPIC_API_KEY")

client = Anthropic(api_key=api_key)
MODEL = "claude-haiku-4-5-20251001"

system_content = """Eres un consultor senior de estimación de proyectos software con amplia experiencia.

Reglas:
- Responde siempre en español.
- Usa terminología técnica sin simplificar en exceso.
- Si ofreces una estimación temporal, indica un rango (optimista / pesimista).
- Si faltan datos para estimar con rigor, pregunta antes de suponer.
- Redacta en prosa; evita listas con viñetas salvo que el usuario las pida."""

user_content = (
    "¿Qué factores debo considerar al estimar un proyecto de migración de base de datos?"
)

try:
    respuesta = client.messages.create(
        model=MODEL,
        system=system_content,
        messages=[{"role": "user", "content": user_content}],
        max_tokens=500,
        temperature=0.3,
    )
except AuthenticationError:
    print("401: revisa ANTHROPIC_API_KEY (clave sk-ant-... válida).")
    raise
except RateLimitError:
    print("429: rate limit o crédito; espera o revisa el panel de Anthropic.")
    raise
except BadRequestError as e:
    print("400:", getattr(e, "message", e))
    raise
except (APIConnectionError, InternalServerError) as e:
    print("Red o servidor Anthropic:", e)
    raise

texto = texto_desde_content(respuesta.content)

print("--- Contenido (bloques text en content) ---")
print(texto)

print("--- stop_reason ---")
print(respuesta.stop_reason)
if respuesta.stop_reason == "max_tokens":
    print("Aviso: respuesta truncada por max_tokens — sube el límite o divide la tarea.")
elif respuesta.stop_reason == "end_turn":
    print("Generación completada de forma natural.")

print("--- Metadatos ---")
print("id:", respuesta.id)
print("model:", respuesta.model)
print("type:", respuesta.type)
print("role:", respuesta.role)

if respuesta.usage:
    u = respuesta.usage
    print("usage:", u.model_dump())
    inp = u.input_tokens
    out = u.output_tokens
    total = inp + out
    print("total_tokens (calculado):", total)
    p = PRICING_HAIKU
    coste = (inp / 1_000_000) * p["input"] + (out / 1_000_000) * p["output"]
    print(f"coste aprox. (USD): ${coste:.6f}")

# Inspección opcional del JSON completo:
# print(respuesta.model_dump_json(indent=2)[:3000])

# --- Opcional: segundo turno con historial manual (stateless) ---
# r1 = client.messages.create(
#     model=MODEL,
#     system=system_content,
#     messages=[{"role": "user", "content": "¿Qué es una API REST?"}],
#     max_tokens=400,
#     temperature=0.3,
# )
# t1 = texto_desde_content(r1.content)
# r2 = client.messages.create(
#     model=MODEL,
#     system=system_content,
#     messages=[
#         {"role": "user", "content": "¿Qué es una API REST?"},
#         {"role": "assistant", "content": t1},
#         {"role": "user", "content": "¿En qué se diferencia de GraphQL?"},
#     ],
#     max_tokens=500,
#     temperature=0.3,
# )
# print(texto_desde_content(r2.content))


## Ejercicio 4 — Google Gemini (SDK `google-genai`)

### Contexto

Google expone Gemini mediante el **Google Gen AI SDK** (`pip install google-genai`), unificado para **Gemini Developer API** (solo API key, usada en el programa por simplicidad) y **Vertex AI** (Google Cloud).

La interfaz principal es **`client.models.generate_content()`**. El SDK **`google-generativeai` está deprecado**; el estándar actual es **`google-genai`**.

### 1. Llamada completa

- **`contents`**: string o estructuras `types.Content` / `types.Part` (multi-turno).
- **`config`**: **`types.GenerateContentConfig`** agrupa `system_instruction`, `temperature`, `max_output_tokens`, `safety_settings`, `thinking_config`, etc.
- **Salida rápida:** `response.text` (primer candidato). Estructura completa en `response.candidates`.

### 2. `system_instruction`

Equivalente conceptual a `instructions` (OpenAI Responses) o `system` (Anthropic): string o **lista de strings** para modularizar reglas.

### 3. `contents` y roles

En historial estructurado, el rol del modelo es **`"model"`**, no `"assistant"`. La API es **stateless**; puedes reconstruir el historial o usar **`client.chats.create()`** como conveniencia (sigue habiendo coste por tokens reenviados).

### 4. Parámetros destacados

`temperature` 0.0–2.0; `max_output_tokens` **opcional** en Gemini (a diferencia de `max_tokens` **obligatorio** en Anthropic). `top_p`, `top_k`, `stop_sequences`, penalizaciones, `seed`, `safety_settings`, `thinking_config` en modelos compatibles.

### 5. Respuesta y `finish_reason`

Valores típicos: `STOP`, `MAX_TOKENS`, `SAFETY`, `RECITATION`, … Los bloqueos por seguridad pueden aparecer aquí **sin** error HTTP.

No hay **ID de petición** ni **timestamp** en la respuesta: añade trazabilidad en tu backend.

### 6. Tokens y coste

`usage_metadata`: `prompt_token_count`, `candidates_token_count`, `thoughts_token_count`, `cached_content_token_count`, `total_token_count`.

**`client.models.count_tokens(model=..., contents=...)`** estima tokens de entrada sin coste (útil para presupuestar).

### 7. Errores y reintentos

`google.genai.errors.ClientError`, `APIError`, `ServerError`. A diferencia de Anthropic, **no** hay reintentos automáticos por defecto: implementa backoff para 429/5xx si lo necesitas.

### 8. Equivalencia rápida

| Concepto | OpenAI Responses | Anthropic Messages | Gemini (`google-genai`) |
|----------|-------------------|--------------------|-------------------------|
| Instrucciones sistema | `instructions` | `system` (top-level) | `config.system_instruction` |
| Entrada principal | `input` | `messages` | `contents` |
| Rol modelo en hilo | `assistant` | `assistant` | **`model`** |
| Texto rápido | `output_text` | bloques `text` | **`response.text`** |
| Tope salida | `max_output_tokens` | **`max_tokens` requerido** | `max_output_tokens` (opc.) |

**Colab:** `%pip install -q google-genai`. Secreto **`GEMINI_API_KEY`** (Google acepta también **`GOOGLE_API_KEY`** en algunos entornos). **Local:** `export GEMINI_API_KEY=...`.

Documentación: [Gemini API](https://ai.google.dev/gemini-api/docs).

### Errores frecuentes

- **401:** clave incorrecta o no cargada.
- **429:** cuotas RPM/TPM/RPD; el tier gratuito es muy limitado.
- **400:** modelo inexistente, `role` incorrecto (`assistant` en lugar de `model`), `contents` vacío.


In [ ]:
# %pip install -q google-genai

import os
from getpass import getpass

from google import genai
from google.genai import types
from google.genai.errors import APIError, ClientError, ServerError


def obtener_api_key_gemini() -> str:
    try:
        from google.colab import userdata  # type: ignore

        key = userdata.get("GEMINI_API_KEY")
        if key:
            return key.strip()
    except ImportError:
        pass
    for env in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
        k = os.environ.get(env, "").strip()
        if k:
            return k
    return getpass("GEMINI_API_KEY (o GOOGLE_API_KEY): ").strip()


# Tarifas ejemplo USD/1M (Gemini 2.5 Flash — revisa https://ai.google.dev/gemini-api/docs/pricing )
PRICING_FLASH = {"input": 0.15, "output": 0.60, "thinking": 3.50}

api_key = obtener_api_key_gemini()
if not api_key:
    raise ValueError("Falta GEMINI_API_KEY o GOOGLE_API_KEY")

client = genai.Client(api_key=api_key)
MODEL = "gemini-2.5-flash"

system_instruction = """Eres un consultor senior de estimación de proyectos software con amplia experiencia.

Reglas:
- Responde siempre en español.
- Usa terminología técnica sin simplificar en exceso.
- Si ofreces una estimación temporal, indica un rango (optimista / pesimista).
- Si faltan datos para estimar con rigor, pregunta antes de suponer.
- Redacta en prosa; evita listas con viñetas salvo que el usuario las pida."""

user_content = (
    "¿Qué factores debo considerar al estimar un proyecto de migración de base de datos?"
)

config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    temperature=0.3,
    max_output_tokens=500,
)

try:
    pre = client.models.count_tokens(model=MODEL, contents=user_content)
    print("--- count_tokens (solo texto usuario) ---")
    print("total_tokens:", pre.total_tokens)
except Exception as exc:
    print("(count_tokens omitido o error):", exc)

try:
    response = client.models.generate_content(
        model=MODEL,
        contents=user_content,
        config=config,
    )
except ClientError as e:
    print("ClientError:", getattr(e, "code", None), getattr(e, "message", e))
    raise
except ServerError as e:
    print("ServerError:", getattr(e, "code", None), getattr(e, "message", e))
    raise
except APIError as e:
    print("APIError:", getattr(e, "code", None), getattr(e, "message", e))
    raise

if not response.candidates:
    raise RuntimeError("La respuesta no contiene candidatos (revisa bloqueos o error de API).")

cand = response.candidates[0]
fr = cand.finish_reason
fr_name = fr.name if fr is not None else None

print("--- Contenido (response.text) ---")
print(response.text)

print("--- finish_reason ---")
print(fr_name)
if fr_name == "MAX_TOKENS":
    print("Aviso: truncado por max_output_tokens.")
elif fr_name == "SAFETY":
    print("Aviso: bloqueo por filtros de seguridad; revisa safety_ratings del candidato.")
elif fr_name == "STOP":
    print("Generación completada (STOP).")

print("--- Metadatos ---")
print("model_version:", response.model_version)

um = response.usage_metadata
if um:
    print("usage_metadata:", um.model_dump())
    inp = um.prompt_token_count or 0
    out = um.candidates_token_count or 0
    think = um.thoughts_token_count or 0
    p = PRICING_FLASH
    coste = (
        (inp / 1_000_000) * p["input"]
        + (out / 1_000_000) * p["output"]
        + (think / 1_000_000) * p["thinking"]
    )
    print(f"coste aprox. (USD): ${coste:.6f}")

# --- Opcional: multi-turno con types.Content (rol model, no assistant) ---
# r1 = client.models.generate_content(
#     model=MODEL,
#     contents="¿Qué es una API REST?",
#     config=config,
# )
# r2 = client.models.generate_content(
#     model=MODEL,
#     contents=[
#         types.Content(role="user", parts=[types.Part(text="¿Qué es una API REST?")]),
#         types.Content(role="model", parts=[types.Part(text=r1.text)]),
#         types.Content(role="user", parts=[types.Part(text="¿En qué se diferencia de GraphQL?")]),
#     ],
#     config=config,
# )
# print(r2.text)


## Ejercicio 5 — Tokens, BPE y presupuesto (tiktoken + Gemini `count_tokens`)

Este bloque **no llama a modelos generativos de pago** salvo la parte opcional de **`count_tokens`** en Gemini (suele ser gratuito; confirma en la doc vigente). Sirve para **ver** cómo el texto se convierte en IDs y cómo eso afecta a **coste y contexto**.

### 1. Tres fases (mapa mental)

1. **UTF‑8:** el texto es bytes (`A` 1 byte, `ñ` 2, muchos emoji 4).
2. **Pre‑tokenización:** reglas que trocean por espacios, puntuación y clases de caracteres.
3. **BPE:** merges aprendidos → secuencia de **token IDs** (lo que cuenta la API de OpenAI en modelos compatibles con `tiktoken`).

### 2. Qué harás en código

- Inspeccionar **`PostgreSQL migration`** token a token (fíjate en espacios y troceos; el número exacto de tokens puede variar con la versión del encoding `o200k_base`).
- Comparar **tamaños de vocabulario** `gpt2` / `cl100k_base` / `o200k_base`.
- Medir **tokens por idioma** en frases semánticamente parecidas (inglés vs español vs francés vs japonés).
- Contar tokens de **código**, **JSON compacto vs *pretty-print***, **espacios iniciales**, **números** y **palabras “opacas”** (`Strawberry`, etc.).
- Opcional: **`client.models.count_tokens`** en **Gemini** para la misma frase en español y comparar **orden de magnitud** con `tiktoken` (son tokenizadores **distintos**).

### 3. Lecturas en el Second Brain

`learnings/second-brain-master-ia/sesiones/sesion-01-llms-setup.md` → sección **«Tokens y tokenización (de texto a números, ventana y coste)»**.

### 4. Errores habituales

- Creer que **1 palabra = 1 token**.
- Usar **`tiktoken`** como conteo exacto para **Anthropic** o **Gemini** (no; aquí solo OpenAI y aproximación mental).
- Enviar **JSON bonito** en contextos enormes sin necesidad (consume tokens de más).


In [ ]:
# %pip install -q tiktoken
# Opcional Gemini (misma clave que ejercicio 4): %pip install -q google-genai

import json
import os
from getpass import getpass

import tiktoken

MODEL_OAI = "gpt-4o-mini"
enc = tiktoken.encoding_for_model(MODEL_OAI)


def banner(title: str) -> None:
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)


# --- 1) Texto → IDs (ejemplo guía sesión 01) ---
banner("1) Texto → token IDs (PostgreSQL migration)")
text = "PostgreSQL migration"
ids = enc.encode(text)
print(f"Text: {text!r}\nToken IDs: {ids}\nCount: {len(ids)} tokens\n")
for tid in ids:
    print(f"  ID {tid:>7} -> {enc.decode([tid])!r}")

# --- 2) UTF-8: bytes por carácter (intuición) ---
banner("2) UTF-8: bytes por carácter (no es el conteo de facturación de la API)")
for s in ["A", "ñ", "🚀", "PostgreSQL"]:
    b = s.encode("utf-8")
    print(f"{s!r:16} chars={len(s)}  bytes={len(b)}")

# --- 3) Vocabularios OpenAI (tiktoken) ---
banner("3) Tamaño de vocabulario por encoding")
for label, name in [
    ("gpt-2", "gpt2"),
    ("gpt-3.5/4", "cl100k_base"),
    ("gpt-4o", "o200k_base"),
]:
    e = tiktoken.get_encoding(name)
    print(f"{label:15} {name:15} n_vocab={e.n_vocab:,}")

# --- 4) Idioma vs tokens (misma idea semántica) ---
banner("4) Idioma: tokens para frases comparables")
rows = [
    ("English", "What are the main risks of migrating to microservices?"),
    ("Spanish", "¿Cuáles son los principales riesgos de migrar a microservicios?"),
    ("French", "Quels sont les principaux risques de la migration vers les microservices?"),
    ("Japanese", "マイクロサービスへの移行の主なリスクは何ですか？"),
]
hdr = f"{'Language':12} {'Chars':>7} {'Tokens':>7} {'Chars/token':>12}"
print(hdr)
print("-" * len(hdr))
for lang, t in rows:
    tok = enc.encode(t)
    ratio = len(t) / len(tok) if tok else 0.0
    print(f"{lang:12} {len(t):>7} {len(tok):>7} {ratio:>12.2f}")

# --- 5) Código y JSON compact vs pretty ---
banner("5) Código y JSON: compacto vs pretty-print")
py_snippet = """def calculate_total(items, tax_rate=0.21):
    subtotal = sum(item.price for item in items)
    return subtotal * (1 + tax_rate)"""
data = {
    "project": "migration",
    "tasks": [
        {"name": "schema-analysis", "hours": 40},
        {"name": "data-transfer", "hours": 80},
        {"name": "testing", "hours": 60},
    ],
}
compact = json.dumps(data)
pretty = json.dumps(data, indent=2)
t_py = len(enc.encode(py_snippet))
t_c = len(enc.encode(compact))
t_p = len(enc.encode(pretty))
print(f"Python snippet: {t_py:>4} tokens, {len(py_snippet):>4} chars")
print(f"JSON compact : {t_c:>4} tokens, {len(compact):>4} chars")
print(f"JSON pretty  : {t_p:>4} tokens, {len(pretty):>4} chars")
print(f"Overhead pretty vs compact: +{t_p - t_c} tokens")

# --- 6) Espacios iniciales ---
banner('6) Espacios: "Hello" vs " Hello" son piezas distintas')
for t in ["Hello", " Hello", "  Hello", "    Hello"]:
    parts = enc.encode(t)
    decoded = [repr(enc.decode([x])) for x in parts]
    print(f"{t!r:12} -> {len(parts)} tokens: {decoded}")

# --- 7) Números ---
banner("7) Números: troceo poco intuitivo (no uses LLM como calculadora)")
for num in ["42", "100", "1000", "12345", "99999", "123456789"]:
    parts = enc.encode(num)
    dec = [repr(enc.decode([x])) for x in parts]
    print(f"{num:>12} -> {len(parts)} token(s): {dec}")

# --- 8) Palabras como una o varias unidades ---
banner("8) Palabras: una unidad opaca o varias piezas")
for w in ["hello", "Strawberry", "tokenization", "microservices", "PostgreSQL"]:
    parts = enc.encode(w)
    dec = [repr(enc.decode([x])) for x in parts]
    label = "1 token" if len(parts) == 1 else f"{len(parts)} tokens"
    print(f"{w:16} -> {label:10} -> {dec}")

# --- 9) Opcional: Gemini count_tokens (tokenizador distinto a OpenAI) ---
banner("9) Opcional: Gemini count_tokens (requiere GEMINI_API_KEY / GOOGLE_API_KEY)")
try:
    from google import genai
except ImportError:
    genai = None  # type: ignore

if genai is None:
    print(
        "Bloque 9 omitido: instala google-genai (ej. ejercicio 4) y define GEMINI_API_KEY para count_tokens."
    )
else:

    def obtener_api_key_gemini() -> str:
        try:
            from google.colab import userdata  # type: ignore

            key = userdata.get("GEMINI_API_KEY")
            if key:
                return key.strip()
        except ImportError:
            pass
        for env in ("GEMINI_API_KEY", "GOOGLE_API_KEY"):
            k = os.environ.get(env, "").strip()
            if k:
                return k
        return getpass("GEMINI_API_KEY (o GOOGLE_API_KEY): ").strip()

    gem_key = obtener_api_key_gemini()
    if not gem_key:
        print("Sin clave: omite este bloque o exporta GEMINI_API_KEY.")
    else:
        client = genai.Client(api_key=gem_key)
        MODEL_GEM = "gemini-2.5-flash"
        probe = "¿Cuáles son los principales riesgos de migrar a microservicios?"
        pre = client.models.count_tokens(model=MODEL_GEM, contents=probe)
        oa = len(enc.encode(probe))
        print("Frase (español):", probe)
        print("Gemini total_tokens:", pre.total_tokens)
        print("OpenAI tiktoken (gpt-4o-mini):", oa)
        print("Nota: son tokenizadores distintos; no esperes igualdad.")

print("\n--- Fin ejercicio 5 ---")
